In [1]:
# Cell 1: Create the project folder and subfolders
import os

# Create project on your Desktop
PROJECT_DIR = os.path.expanduser("~/Desktop/grant_monitor_app")
TEMPLATES_DIR = os.path.join(PROJECT_DIR, "templates")

os.makedirs(TEMPLATES_DIR, exist_ok=True)

print(f"✅ Project folder: {PROJECT_DIR}")
print(f"✅ Templates folder: {TEMPLATES_DIR}")
print(f"\nContents:")
for item in os.listdir(PROJECT_DIR):
    print(f"   📁 {item}" if os.path.isdir(os.path.join(PROJECT_DIR, item)) else f"   📄 {item}")

✅ Project folder: /Users/kschro10/Desktop/grant_monitor_app
✅ Templates folder: /Users/kschro10/Desktop/grant_monitor_app/templates

Contents:
   📄 requirements.txt
   📁 templates


In [2]:
# Cell 2: Write requirements.txt

requirements = """flask
pywebview
requests
beautifulsoup4
feedparser
pandas
lxml
openpyxl
python-dateutil
"""

with open(os.path.join(PROJECT_DIR, "requirements.txt"), "w") as f:
    f.write(requirements)

print("✅ requirements.txt written")

✅ requirements.txt written


In [3]:
# Cell 3: Install all dependencies (run once)

import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "flask", "pywebview", "requests", "beautifulsoup4",
    "feedparser", "pandas", "lxml", "openpyxl", "python-dateutil"
])

print("✅ All dependencies installed")

✅ All dependencies installed


In [4]:
# Fix: Set PROJECT_DIR to your actual project location

import os

PROJECT_DIR = "/Users/kschro10/anaconda_projects/1b55a599-cf36-4af0-9a0e-86dc28da2d79"
TEMPLATES_DIR = os.path.join(PROJECT_DIR, "templates")

# Make sure templates folder exists
os.makedirs(TEMPLATES_DIR, exist_ok=True)

print(f"✅ PROJECT_DIR: {PROJECT_DIR}")
print(f"✅ TEMPLATES_DIR: {TEMPLATES_DIR}")
print(f"   Exists: {os.path.exists(PROJECT_DIR)}")

✅ PROJECT_DIR: /Users/kschro10/anaconda_projects/1b55a599-cf36-4af0-9a0e-86dc28da2d79
✅ TEMPLATES_DIR: /Users/kschro10/anaconda_projects/1b55a599-cf36-4af0-9a0e-86dc28da2d79/templates
   Exists: True


In [5]:
requirements = """flask
pywebview
requests
beautifulsoup4
feedparser
pandas
lxml
openpyxl
python-dateutil
"""

with open(os.path.join(PROJECT_DIR, "requirements.txt"), "w") as f:
    f.write(requirements)

print("✅ requirements.txt written")

✅ requirements.txt written


In [6]:
# Cell 4: Write grant_engine.py — all your grant scanning logic

grant_engine_code = r'''"""
grant_engine.py — All grant scanning, DB, keyword, and export logic.
Consolidated from the Jupyter notebook into a single importable module.
"""

import requests
import feedparser
import sqlite3
import hashlib
import json
import re
import time
import logging
from bs4 import BeautifulSoup
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from contextlib import contextmanager
import pandas as pd
from datetime import datetime, timedelta, timezone

try:
    from dateutil.relativedelta import relativedelta
except ImportError:
    relativedelta = None

try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter
    from openpyxl.worksheet.datavalidation import DataValidation
    HAS_OPENPYXL = True
except ImportError:
    HAS_OPENPYXL = False


# ==============================================================
# UTILITIES
# ==============================================================

def utc_now() -> datetime:
    return datetime.now(timezone.utc).replace(tzinfo=None)


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("grant_monitor.log"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("grant_monitor")

CONFIG = {
    "db_path": "grants.db",
    "request_delay_seconds": 2,
    "request_timeout_seconds": 30,
    "user_agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

DEFAULT_KEYWORDS = [
    "cybersecurity", "AI", "experiential learning", "medical technology",
    "data science", "machine learning", "artificial intelligence",
    "STEM education", "novel education", "workforce",
    "workforce development", "quantum",
]


# ==============================================================
# DATABASE
# ==============================================================

@contextmanager
def get_db(db_path: str = None):
    if db_path is None:
        db_path = CONFIG["db_path"]
    conn = sqlite3.connect(db_path, timeout=10)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA busy_timeout=5000")
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


@dataclass
class Grant:
    title: str
    url: str
    source: str
    description: str = ""
    deadline: Optional[str] = None
    posted_date: Optional[str] = None
    funding_amount: Optional[str] = None
    matched_keywords: list = field(default_factory=list)
    content_hash: str = ""
    first_seen: str = ""
    last_seen: str = ""
    is_new: bool = True

    def compute_hash(self):
        raw = f"{self.title}|{self.description}|{self.deadline}".lower().strip()
        self.content_hash = hashlib.sha256(raw.encode()).hexdigest()
        return self.content_hash


def init_db(db_path: str = None):
    if db_path is None:
        db_path = CONFIG["db_path"]
    with get_db(db_path) as conn:
        c = conn.cursor()
        c.execute("""
            CREATE TABLE IF NOT EXISTS grants (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL, url TEXT NOT NULL, source TEXT NOT NULL,
                description TEXT, deadline TEXT, posted_date TEXT,
                funding_amount TEXT, matched_keywords TEXT,
                content_hash TEXT UNIQUE, first_seen TEXT NOT NULL,
                last_seen TEXT NOT NULL, notified INTEGER DEFAULT 0
            )
        """)
        c.execute("""
            CREATE TABLE IF NOT EXISTS scan_log (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                source TEXT NOT NULL, scan_time TEXT NOT NULL,
                grants_found INTEGER DEFAULT 0, new_grants INTEGER DEFAULT 0,
                updated_grants INTEGER DEFAULT 0, status TEXT DEFAULT 'success',
                error_message TEXT
            )
        """)
        c.execute("""
            CREATE TABLE IF NOT EXISTS keywords (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                keyword TEXT UNIQUE NOT NULL,
                added_date TEXT NOT NULL, active INTEGER DEFAULT 1
            )
        """)
        now = utc_now().strftime("%Y-%m-%dT%H:%M:%S")
        for kw in DEFAULT_KEYWORDS:
            c.execute(
                "INSERT OR IGNORE INTO keywords (keyword, added_date) VALUES (?, ?)",
                (kw, now),
            )
    logger.info("Database initialized.")


def upsert_grant(grant: Grant, db_path: str = None) -> str:
    if db_path is None:
        db_path = CONFIG["db_path"]
    now = utc_now().strftime("%Y-%m-%dT%H:%M:%S")
    grant.compute_hash()
    grant.last_seen = now

    with get_db(db_path) as conn:
        c = conn.cursor()
        c.execute("SELECT id, content_hash FROM grants WHERE url = ?", (grant.url,))
        row = c.fetchone()

        if row is None:
            grant.first_seen = now
            c.execute("""
                INSERT INTO grants
                (title, url, source, description, deadline, posted_date,
                 funding_amount, matched_keywords, content_hash, first_seen, last_seen)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                grant.title, grant.url, grant.source, grant.description,
                grant.deadline, grant.posted_date, grant.funding_amount,
                json.dumps(grant.matched_keywords), grant.content_hash,
                grant.first_seen, grant.last_seen,
            ))
            return "new"
        elif row[1] != grant.content_hash:
            c.execute("""
                UPDATE grants SET title=?, description=?, deadline=?, posted_date=?,
                    funding_amount=?, matched_keywords=?, content_hash=?,
                    last_seen=?, notified=0
                WHERE url=?
            """, (
                grant.title, grant.description, grant.deadline,
                grant.posted_date, grant.funding_amount,
                json.dumps(grant.matched_keywords), grant.content_hash,
                grant.last_seen, grant.url,
            ))
            return "updated"
        else:
            c.execute("UPDATE grants SET last_seen=? WHERE url=?", (now, grant.url))
            return "unchanged"


def log_scan(source, grants_found, new_grants, updated_grants,
             status="success", error_message=None):
    with get_db() as conn:
        conn.execute("""
            INSERT INTO scan_log
            (source, scan_time, grants_found, new_grants, updated_grants, status, error_message)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (source, utc_now().strftime("%Y-%m-%dT%H:%M:%S"), grants_found,
              new_grants, updated_grants, status, error_message))


# ==============================================================
# KEYWORDS
# ==============================================================

def get_active_keywords(db_path: str = None) -> list:
    if db_path is None:
        db_path = CONFIG["db_path"]
    with get_db(db_path) as conn:
        c = conn.cursor()
        c.execute("SELECT keyword FROM keywords WHERE active = 1")
        return [row[0] for row in c.fetchall()]


def add_keyword(kw: str):
    now = utc_now().strftime("%Y-%m-%dT%H:%M:%S")
    with get_db() as conn:
        conn.execute(
            "INSERT OR IGNORE INTO keywords (keyword, added_date, active) VALUES (?, ?, 1)",
            (kw, now),
        )
        conn.execute(
            "UPDATE keywords SET active = 1 WHERE keyword = ? AND active = 0", (kw,)
        )


def remove_keyword(kw: str):
    with get_db() as conn:
        conn.execute("UPDATE keywords SET active = 0 WHERE keyword = ?", (kw,))


def match_keywords(text: str, keywords: list = None) -> list:
    if keywords is None:
        keywords = get_active_keywords()
    text_lower = text.lower()
    matched = []
    for kw in keywords:
        if kw == "AI":
            if re.search(r'\bAI\b', text):
                matched.append(kw)
        else:
            if re.search(r'\b' + re.escape(kw.lower()) + r'\b', text_lower):
                matched.append(kw)
    return matched


def passes_filter(title: str, description: str) -> list:
    return match_keywords(f"{title} {description}")


# ==============================================================
# HTTP SESSION
# ==============================================================

def get_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({
        "User-Agent": CONFIG["user_agent"],
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    })
    return s


SESSION = get_session()


def safe_request(url: str, **kwargs):
    try:
        time.sleep(CONFIG["request_delay_seconds"])
        resp = SESSION.get(url, timeout=CONFIG["request_timeout_seconds"], **kwargs)
        resp.raise_for_status()
        return resp
    except requests.RequestException as e:
        logger.error(f"Request failed for {url}: {e}")
        return None


# ==============================================================
# SCRAPERS
# ==============================================================

def scrape_grants_gov(keywords=None):
    if keywords is None:
        keywords = get_active_keywords()
    grants = []
    base_url = "https://apply07.grants.gov/grantsws/rest/opportunities/search/"
    for kw in keywords:
        payload = {
            "keyword": kw, "oppStatuses": "forecasted|posted",
            "sortBy": "openDate|desc", "rows": 50, "startRecordNum": 0,
        }
        try:
            time.sleep(CONFIG["request_delay_seconds"])
            resp = requests.post(base_url, json=payload,
                                 headers={"Content-Type": "application/json"},
                                 timeout=CONFIG["request_timeout_seconds"])
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            logger.error(f"Grants.gov API error for '{kw}': {e}")
            continue
        opportunities = data.get("oppHits", [])
        logger.info(f"Grants.gov: '{kw}' returned {len(opportunities)} results")
        for opp in opportunities:
            title = opp.get("title", "")
            description = opp.get("description", title)
            url = f"https://www.grants.gov/search-results-detail/{opp.get('id', '')}"
            matched = passes_filter(title, description)
            if matched:
                g = Grant(
                    title=title, url=url, source="grants.gov",
                    description=description[:2000],
                    deadline=opp.get("closeDate", ""),
                    posted_date=opp.get("openDate", ""),
                    funding_amount=opp.get("awardCeiling", ""),
                    matched_keywords=matched,
                )
                grants.append(g)
    seen = set()
    unique = []
    for g in grants:
        if g.url not in seen:
            seen.add(g.url)
            unique.append(g)
    logger.info(f"Grants.gov total unique matches: {len(unique)}")
    return unique


def scrape_nsf():
    feeds = [
        "https://www.nsf.gov/rss/rss_www_funding.xml",
        "https://www.nsf.gov/rss/rss_www_funding_pgm_annc_cise.xml",
        "https://www.nsf.gov/rss/rss_www_funding_pgm_annc_ehr.xml",
    ]
    grants = []
    for feed_url in feeds:
        try:
            time.sleep(CONFIG["request_delay_seconds"])
            feed = feedparser.parse(feed_url)
            logger.info(f"NSF feed: {len(feed.entries)} entries from {feed_url}")
        except Exception as e:
            logger.error(f"NSF feed error ({feed_url}): {e}")
            continue
        for entry in feed.entries:
            title = entry.get("title", "")
            summary = entry.get("summary", "")
            link = entry.get("link", "")
            published = entry.get("published", "")
            matched = passes_filter(title, summary)
            if matched:
                g = Grant(title=title, url=link, source="nsf.gov",
                          description=summary[:2000], posted_date=published,
                          matched_keywords=matched)
                grants.append(g)
    seen = set()
    unique = []
    for g in grants:
        if g.url not in seen:
            seen.add(g.url)
            unique.append(g)
    logger.info(f"NSF total unique matches: {len(unique)}")
    return unique


def scrape_nih_reporter():
    grants = []
    keywords = get_active_keywords()
    url = "https://api.reporter.nih.gov/v2/projects/search"
    search_text = " OR ".join(
        [kw for kw in keywords if kw.lower() in
         ("medical technology", "ai", "artificial intelligence",
          "data science", "machine learning", "cybersecurity")]
    )
    payload = {
        "criteria": {
            "advanced_text_search": {
                "operator": "or", "search_field": "projecttitle,terms",
                "search_text": search_text,
            },
            "fiscal_years": [2025, 2026],
            "newly_added_projects_only": True,
        },
        "offset": 0, "limit": 50,
        "sort_field": "project_start_date", "sort_order": "desc",
    }
    try:
        time.sleep(CONFIG["request_delay_seconds"])
        resp = requests.post(url, json=payload, timeout=CONFIG["request_timeout_seconds"])
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        logger.error(f"NIH Reporter API error: {e}")
        return grants
    results = data.get("results", [])
    logger.info(f"NIH Reporter: {len(results)} results")
    for proj in results:
        title = proj.get("project_title", "")
        abstract = proj.get("abstract_text", "") or ""
        link = f"https://reporter.nih.gov/project-details/{proj.get('application_id', '')}"
        matched = passes_filter(title, abstract)
        if matched:
            award = proj.get("award_amount")
            g = Grant(
                title=title.strip(), url=link, source="nih_reporter",
                description=abstract[:2000],
                posted_date=proj.get("project_start_date", ""),
                funding_amount=str(award) if award else "",
                matched_keywords=matched,
            )
            grants.append(g)
    logger.info(f"NIH Reporter matches: {len(grants)}")
    return grants


def scrape_rss_source(feed_url, source_name):
    grants = []
    try:
        time.sleep(CONFIG["request_delay_seconds"])
        feed = feedparser.parse(feed_url)
        logger.info(f"{source_name}: {len(feed.entries)} entries")
    except Exception as e:
        logger.error(f"{source_name} feed error: {e}")
        return grants
    for entry in feed.entries:
        title = entry.get("title", "")
        summary = entry.get("summary", entry.get("description", ""))
        link = entry.get("link", "")
        published = entry.get("published", entry.get("updated", ""))
        matched = passes_filter(title, summary)
        if matched:
            g = Grant(title=title, url=link, source=source_name,
                      description=summary[:2000], posted_date=published,
                      matched_keywords=matched)
            grants.append(g)
    logger.info(f"{source_name} matches: {len(grants)}")
    return grants


ADDITIONAL_RSS_SOURCES = [
    {"url": "https://www.ed.gov/feed", "name": "ed.gov"},
    {"url": "https://www.sbir.gov/rss-feed", "name": "sbir.gov"},
    {"url": "https://www.darpa.mil/rss", "name": "darpa.mil"},
]


# ==============================================================
# FULL SCAN
# ==============================================================

def run_full_scan() -> dict:
    logger.info("=" * 60)
    logger.info("STARTING FULL GRANT SCAN")
    logger.info(f"Active keywords: {get_active_keywords()}")
    logger.info("=" * 60)

    summary = {
        "scan_time": utc_now().isoformat(),
        "sources_scanned": 0, "total_found": 0,
        "new": 0, "updated": 0, "unchanged": 0, "errors": [],
    }
    all_grants = []

    try:
        results = scrape_grants_gov()
        all_grants.extend(results)
        log_scan("grants.gov", len(results), 0, 0)
        summary["sources_scanned"] += 1
    except Exception as e:
        logger.error(f"Grants.gov scraper failed: {e}")
        log_scan("grants.gov", 0, 0, 0, "error", str(e))
        summary["errors"].append(f"grants.gov: {e}")

    try:
        results = scrape_nsf()
        all_grants.extend(results)
        log_scan("nsf.gov", len(results), 0, 0)
        summary["sources_scanned"] += 1
    except Exception as e:
        logger.error(f"NSF scraper failed: {e}")
        log_scan("nsf.gov", 0, 0, 0, "error", str(e))
        summary["errors"].append(f"nsf.gov: {e}")

    try:
        results = scrape_nih_reporter()
        all_grants.extend(results)
        log_scan("nih_reporter", len(results), 0, 0)
        summary["sources_scanned"] += 1
    except Exception as e:
        logger.error(f"NIH Reporter scraper failed: {e}")
        log_scan("nih_reporter", 0, 0, 0, "error", str(e))
        summary["errors"].append(f"nih_reporter: {e}")

    for src in ADDITIONAL_RSS_SOURCES:
        try:
            results = scrape_rss_source(src["url"], src["name"])
            all_grants.extend(results)
            log_scan(src["name"], len(results), 0, 0)
            summary["sources_scanned"] += 1
        except Exception as e:
            logger.error(f"{src['name']} scraper failed: {e}")
            log_scan(src["name"], 0, 0, 0, "error", str(e))
            summary["errors"].append(f"{src['name']}: {e}")

    summary["total_found"] = len(all_grants)
    for grant in all_grants:
        result = upsert_grant(grant)
        summary[result] = summary.get(result, 0) + 1

    logger.info("=" * 60)
    logger.info(f"SCAN COMPLETE: {summary}")
    logger.info("=" * 60)
    return summary


# ==============================================================
# REPORTS
# ==============================================================

def get_grants_df(since_days: int = 7, source_filter: str = None) -> pd.DataFrame:
    cutoff = (utc_now() - timedelta(days=since_days)).strftime("%Y-%m-%dT%H:%M:%S")
    with get_db() as conn:
        df = pd.read_sql_query("""
            SELECT title, url, source, description, deadline,
                   posted_date, funding_amount, matched_keywords,
                   first_seen, last_seen,
                   CASE WHEN first_seen >= ? THEN 'NEW' ELSE 'UPDATED' END as status
            FROM grants
            WHERE first_seen >= ? OR (last_seen >= ? AND notified = 0)
            ORDER BY first_seen DESC
        """, conn, params=(cutoff, cutoff, cutoff))
    if not df.empty:
        df["matched_keywords"] = df["matched_keywords"].apply(
            lambda x: ", ".join(json.loads(x)) if x else ""
        )
    if source_filter and source_filter != "all":
        df = df[df["source"] == source_filter]
    return df


def get_all_grants_df() -> pd.DataFrame:
    with get_db() as conn:
        df = pd.read_sql_query("""
            SELECT title, url, source, description, deadline,
                   posted_date, funding_amount, matched_keywords,
                   first_seen, last_seen
            FROM grants ORDER BY first_seen DESC
        """, conn)
    if not df.empty:
        df["matched_keywords"] = df["matched_keywords"].apply(
            lambda x: ", ".join(json.loads(x)) if x else ""
        )
    return df


def export_csv(since_days: int = 7) -> str:
    df = get_grants_df(since_days)
    filename = f"grant_report_{utc_now().strftime('%Y%m%d')}.csv"
    df.to_csv(filename, index=False)
    logger.info(f"CSV exported: {filename}")
    return filename


# ==============================================================
# EXCEL EXPORT
# ==============================================================

def parse_deadline(raw: str) -> Optional[datetime]:
    if not raw or raw.strip() == "":
        return None
    formats = [
        "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d", "%m/%d/%Y",
        "%B %d, %Y", "%b %d, %Y", "%m-%d-%Y", "%Y%m%d",
    ]
    cleaned = raw.strip().split("+")[0].split("Z")[0]
    for fmt in formats:
        try:
            return datetime.strptime(cleaned, fmt)
        except ValueError:
            continue
    return None


def generate_grant_excel(output_filename: str = None) -> str:
    if not HAS_OPENPYXL:
        raise ImportError("openpyxl is required. pip install openpyxl")
    if output_filename is None:
        output_filename = f"Grant_Tracker_{utc_now().strftime('%Y%m%d')}.xlsx"

    df = get_all_grants_df()
    if df.empty:
        return None

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Grant Tracker"
    ws.freeze_panes = "A2"

    HEADER_FILL = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
    HEADER_FONT = Font(name="Calibri", bold=True, color="FFFFFF", size=11)
    THIN_BORDER = Border(
        left=Side(style="thin", color="B4C6E7"),
        right=Side(style="thin", color="B4C6E7"),
        top=Side(style="thin", color="B4C6E7"),
        bottom=Side(style="thin", color="B4C6E7"),
    )

    columns = [
        ("Grant / Program", 50), ("Source", 16), ("Keywords", 28),
        ("Deadline", 16), ("Funding", 18), ("URL", 50),
        ("First Seen", 20), ("Status", 12),
    ]
    for col_idx, (header, width) in enumerate(columns, 1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = THIN_BORDER
        ws.column_dimensions[get_column_letter(col_idx)].width = width

    for row_idx, (_, row) in enumerate(df.iterrows(), 2):
        dl = parse_deadline(row.get("deadline", ""))
        days_left = (dl - utc_now()).days if dl else None
        status = "NEW" if row_idx <= 10 else ""

        values = [
            row["title"], row["source"], row["matched_keywords"],
            dl.strftime("%b %d, %Y") if dl else "TBD",
            row.get("funding_amount", ""),
            row["url"], row["first_seen"], status,
        ]
        for col_idx, val in enumerate(values, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=val)
            cell.border = THIN_BORDER
            cell.alignment = Alignment(wrap_text=True, vertical="top")
            if col_idx == 6:
                cell.font = Font(color="0563C1", underline="single", size=9)
                cell.hyperlink = str(val)
            if col_idx == 4 and days_left is not None and 0 < days_left <= 30:
                cell.font = Font(color="C62828", bold=True)

    ws.auto_filter.ref = f"A1:{get_column_letter(len(columns))}{len(df) + 1}"
    wb.save(output_filename)
    logger.info(f"Excel saved: {output_filename}")
    return output_filename
'''

with open(os.path.join(PROJECT_DIR, "grant_engine.py"), "w") as f:
    f.write(grant_engine_code)

print(f"✅ grant_engine.py written ({len(grant_engine_code):,} characters)")

✅ grant_engine.py written (23,047 characters)


In [7]:
# Cell 5: Write the HTML user interface

html_code = r'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Grant Monitor</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
                         "Helvetica Neue", Arial, sans-serif;
            background: #f0f2f5; color: #1a1a2e; min-height: 100vh;
        }.header {
            background: linear-gradient(135deg, #1a237e 0%, #1565c0 100%);
            color: white; padding: 20px 28px;
            display: flex; justify-content: space-between; align-items: center;
            box-shadow: 0 2px 8px rgba(0,0,0,0.15);
        }.header h1 { font-size: 1.4em; font-weight: 600; }.header p { opacity: 0.8; font-size: 0.85em; margin-top: 2px; }.header-right { text-align: right; font-size: 0.8em; opacity: 0.7; }.container { max-width: 1100px; margin: 0 auto; padding: 20px; }.card {
            background: white; border-radius: 12px; padding: 20px 24px;
            margin-bottom: 16px; box-shadow: 0 1px 4px rgba(0,0,0,0.06);
            border: 1px solid #e8ecf0;
        }.card h2 {
            font-size: 1.05em; color: #1a237e; margin-bottom: 14px;
            display: flex; align-items: center; gap: 8px;
        }.keyword-row {
            display: flex; gap: 8px; margin-bottom: 10px;
            flex-wrap: wrap; align-items: center;
        }.keyword-row input[type="text"] {
            flex: 1; min-width: 220px; padding: 8px 14px;
            border: 1px solid #ccd5e0; border-radius: 8px;
            font-size: 0.9em; outline: none; transition: border-color 0.2s;
        }.keyword-row input[type="text"]:focus { border-color: #1565c0; }.keyword-badges { display: flex; flex-wrap: wrap; gap: 6px; margin-top: 8px; }.badge {
            display: inline-flex; align-items: center; gap: 6px;
            background: #e3f2fd; color: #1565c0; padding: 5px 12px;
            border-radius: 20px; font-size: 0.82em; font-weight: 500;
        }.badge.remove-kw {
            cursor: pointer; color: #c62828; font-weight: bold;
            font-size: 1.1em; line-height: 1; margin-left: 2px;
        }.badge.remove-kw:hover { color: #b71c1c; }.controls-row {
            display: flex; gap: 10px; flex-wrap: wrap;
            align-items: center; margin-bottom: 12px;
        }.controls-row label { font-size: 0.85em; color: #555; font-weight: 500; }.controls-row select,.controls-row input[type="range"] {
            padding: 6px 10px; border: 1px solid #ccd5e0;
            border-radius: 8px; font-size: 0.85em;
        }.btn {
            display: inline-flex; align-items: center; gap: 6px;
            padding: 9px 18px; border: none; border-radius: 8px;
            font-size: 0.88em; font-weight: 600; cursor: pointer;
            transition: all 0.2s; white-space: nowrap;
        }.btn:hover { filter: brightness(0.92); transform: translateY(-1px); }.btn:active { transform: translateY(0); }.btn:disabled { opacity: 0.5; cursor: not-allowed; filter: none; transform: none; }.btn-primary { background: #1565c0; color: white; }.btn-success { background: #2e7d32; color: white; }.btn-info    { background: #00838f; color: white; }.btn-warn    { background: #e65100; color: white; }.btn-sm      { padding: 6px 12px; font-size: 0.82em; }.btn-group   { display: flex; gap: 10px; flex-wrap: wrap; }.status-banner {
            padding: 12px 16px; border-radius: 8px; font-size: 0.88em;
            margin-bottom: 12px; display: none; border-left: 4px solid;
        }.status-banner.info    { background: #e3f2fd; border-color: #1565c0; color: #0d47a1; display: block; }.status-banner.success { background: #e8f5e9; border-color: #2e7d32; color: #1b5e20; display: block; }.status-banner.error   { background: #ffebee; border-color: #c62828; color: #b71c1c; display: block; }.status-banner.warn    { background: #fff3e0; border-color: #e65100; color: #bf360c; display: block; }.grant-card {
            border: 1px solid #e0e4ea; border-radius: 10px; padding: 16px;
            margin-bottom: 10px; background: #fafbfc; transition: box-shadow 0.2s;
        }.grant-card:hover { box-shadow: 0 2px 8px rgba(0,0,0,0.07); }.grant-card.badges { margin-bottom: 8px; }.grant-card.badges span {
            display: inline-block; padding: 2px 10px; border-radius: 4px;
            font-size: 0.75em; font-weight: 700; margin-right: 4px;
        }.badge-new     { background: #28a745; color: white; }.badge-updated { background: #fd7e14; color: white; }.badge-source  { background: #6c757d; color: white; }.grant-card h3 { margin: 6px 0 4px 0; font-size: 1em; }.grant-card h3 a { color: #1565c0; text-decoration: none; }.grant-card h3 a:hover { text-decoration: underline; }.grant-card.desc { color: #666; font-size: 0.85em; margin: 4px 0; line-height: 1.4; }.grant-card.meta { font-size: 0.82em; color: #444; margin-top: 6px; }.spinner {
            display: inline-block; width: 18px; height: 18px;
            border: 3px solid rgba(255,255,255,0.3); border-top-color: white;
            border-radius: 50%; animation: spin 0.8s linear infinite; vertical-align: middle;
        }
        @keyframes spin { to { transform: rotate(360deg); } }.days-display {
            display: inline-block; min-width: 36px; text-align: center;
            font-weight: 700; color: #1565c0;
        }.stats-row { display: flex; gap: 16px; flex-wrap: wrap; margin-bottom: 12px; }.stat-box {
            background: #f5f7fa; border-radius: 8px; padding: 10px 18px;
            text-align: center; min-width: 100px;
        }.stat-box.num { font-size: 1.5em; font-weight: 700; color: #1a237e; }.stat-box.label {
            font-size: 0.75em; color: #888; text-transform: uppercase; letter-spacing: 0.5px;
        }.empty-state { text-align: center; padding: 40px; color: #999; }.empty-state.icon { font-size: 3em; margin-bottom: 10px; }
    </style>
</head>
<body>

<div class="header">
    <div>
        <h1>📋 Grant Monitor</h1>
        <p>Automated grant discovery — Grants.gov, NSF, NIH Reporter & more</p>
    </div>
    <div class="header-right" id="clock"></div>
</div>

<div class="container">
    <div class="status-banner" id="statusBanner"></div>

    <!-- KEYWORD MANAGEMENT -->
    <div class="card">
        <h2>🏷️ Keyword Management</h2>
        <div class="keyword-row">
            <input type="text" id="keywordInput"
                   placeholder="Enter a keyword (e.g., quantum computing)"
                   onkeydown="if(event.key==='Enter') addKeyword()">
            <button class="btn btn-success btn-sm" onclick="addKeyword()">➕ Add</button>
        </div>
        <div class="keyword-badges" id="keywordBadges"></div>
    </div>

    <!-- SCAN CONTROLS -->
    <div class="card">
        <h2>⚙️ Scan Controls</h2>
        <div class="controls-row">
            <label>Show last</label>
            <input type="range" id="daysSlider" min="1" max="90" value="7"
                   oninput="document.getElementById('daysVal').textContent = this.value">
            <span class="days-display" id="daysVal">7</span>
            <label>days</label>
            <label style="margin-left: 20px;">Source:</label>
            <select id="sourceFilter">
                <option value="all">All Sources</option>
                <option value="grants.gov">Grants.gov</option>
                <option value="nsf.gov">NSF</option>
                <option value="nih_reporter">NIH Reporter</option>
                <option value="ed.gov">ED.gov</option>
                <option value="sbir.gov">SBIR</option>
            </select>
        </div>
        <div class="btn-group">
            <button class="btn btn-primary" id="scanBtn" onclick="runScan()">
                🔄 Refresh / Scan Now
            </button>
            <button class="btn btn-info" onclick="viewSaved()">
                📋 View Saved Grants
            </button>
            <button class="btn btn-success" onclick="exportExcel()">
                📊 Export Excel
            </button>
            <button class="btn btn-warn" onclick="exportCsv()">
                📄 Export CSV
            </button>
        </div>
    </div>

    <!-- RESULTS -->
    <div class="card">
        <h2>📊 Results</h2>
        <div class="stats-row" id="statsRow" style="display:none;">
            <div class="stat-box"><div class="num" id="statTotal">0</div><div class="label">Total</div></div>
            <div class="stat-box"><div class="num" id="statNew">0</div><div class="label">New</div></div>
            <div class="stat-box"><div class="num" id="statUpdated">0</div><div class="label">Updated</div></div>
            <div class="stat-box"><div class="num" id="statSources">0</div><div class="label">Sources</div></div>
        </div>
        <div id="resultsArea">
            <div class="empty-state">
                <div class="icon">🔍</div>
                <p>Click <strong>Refresh / Scan Now</strong> to discover grants,<br>
                   or <strong>View Saved Grants</strong> to see previous results.</p>
            </div>
        </div>
    </div>
</div>

<script>
    function updateClock() {
        document.getElementById('clock').innerHTML =
            new Date().toUTCString().replace('GMT', 'UTC');
    }
    setInterval(updateClock, 1000);
    updateClock();

    function showStatus(msg, type) {
        const el = document.getElementById('statusBanner');
        el.className = 'status-banner ' + type;
        el.innerHTML = msg;
    }

    async function loadKeywords() {
        const resp = await fetch('/api/keywords');
        const data = await resp.json();
        const container = document.getElementById('keywordBadges');
        if (data.keywords.length === 0) {
            container.innerHTML = '<span style="color:#999;">No active keywords.</span>';
            return;
        }
        container.innerHTML = data.keywords.map(kw =>
            `<span class="badge">${kw}<span class="remove-kw" onclick="removeKeyword('${kw.replace(/'/g, "\\'")}')" title="Remove">✕</span></span>`
        ).join('');
    }

    async function addKeyword() {
        const input = document.getElementById('keywordInput');
        const kw = input.value.trim();
        if (!kw) return;
        const resp = await fetch('/api/keywords', {
            method: 'POST',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({keyword: kw}),
        });
        const data = await resp.json();
        if (data.status === 'ok') {
            showStatus('✅ Keyword added: <strong>' + kw + '</strong>', 'success');
            input.value = '';
            loadKeywords();
        } else {
            showStatus('⚠️ ' + data.message, 'error');
        }
    }

    async function removeKeyword(kw) {
        const resp = await fetch('/api/keywords', {
            method: 'DELETE',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({keyword: kw}),
        });
        const data = await resp.json();
        if (data.status === 'ok') {
            showStatus('🗑️ Keyword removed: <strong>' + kw + '</strong>', 'warn');
            loadKeywords();
        }
    }

    async function runScan() {
        const btn = document.getElementById('scanBtn');
        btn.disabled = true;
        btn.innerHTML = '<span class="spinner"></span> Scanning...';
        showStatus('🔄 <strong>Scanning all sources...</strong> This may take 1–2 minutes.', 'info');
        document.getElementById('resultsArea').innerHTML =
            '<div class="empty-state"><div class="spinner" style="width:40px;height:40px;border-width:4px;border-color:rgba(21,101,192,0.2);border-top-color:#1565c0;"></div><p style="margin-top:12px;">Scanning grant sources...</p></div>';
        try {
            const resp = await fetch('/api/scan', {method: 'POST'});
            const data = await resp.json();
            document.getElementById('statsRow').style.display = 'flex';
            document.getElementById('statTotal').textContent = data.summary.total_found;
            document.getElementById('statNew').textContent = data.summary.new;
            document.getElementById('statUpdated').textContent = data.summary.updated;
            document.getElementById('statSources').textContent = data.summary.sources_scanned;
            let errHtml = '';
            if (data.summary.errors && data.summary.errors.length > 0) {
                errHtml = '<br>⚠️ Errors: ' + data.summary.errors.join(', ');
            }
            showStatus(
                '✅ <strong>Scan complete</strong> — ' + data.summary.sources_scanned + ' sources | ' +
                '<strong>' + data.summary.new + '</strong> new | <strong>' + data.summary.updated + '</strong> updated | ' +
                data.summary.total_found + ' total' + errHtml, 'success'
            );
            await viewSaved();
        } catch (e) {
            showStatus('❌ Scan failed: ' + e.message, 'error');
            document.getElementById('resultsArea').innerHTML =
                '<div class="empty-state"><div class="icon">❌</div><p>Scan failed. Check logs.</p></div>';
        }
        btn.disabled = false;
        btn.innerHTML = '🔄 Refresh / Scan Now';
    }

    async function viewSaved() {
        const days = document.getElementById('daysSlider').value;
        const source = document.getElementById('sourceFilter').value;
        const resp = await fetch('/api/grants?days=' + days + '&source=' + source);
        const data = await resp.json();
        renderGrants(data.grants);
    }

    function renderGrants(grants) {
        const area = document.getElementById('resultsArea');
        if (!grants || grants.length === 0) {
            area.innerHTML = '<div class="empty-state"><div class="icon">📭</div>' +
                '<p>No grants found for this period and filter.</p></div>';
            return;
        }
        let html = '';
        for (const g of grants) {
            const badgeClass = g.status === 'NEW' ? 'badge-new' : 'badge-updated';
            const desc = g.description ? (g.description.length > 300 ?
                g.description.substring(0, 300) + '...' : g.description) : '';
            const deadline = g.deadline ? '<br>⏰ Deadline: ' + g.deadline : '';
            const funding = g.funding_amount ? '<br>💰 Funding: ' + g.funding_amount : '';
            html += '<div class="grant-card">' +
                '<div class="badges">' +
                '<span class="' + badgeClass + '">' + g.status + '</span>' +
                '<span class="badge-source">' + g.source + '</span></div>' +
                '<h3><a href="' + g.url + '" target="_blank">' + g.title + '</a></h3>' +
                '<p class="desc">' + desc + '</p>' +
                '<p class="meta">🏷️ Keywords: <em>' + g.matched_keywords + '</em>' +
                deadline + funding + '</p></div>';
        }
        area.innerHTML = html;
    }

    async function exportExcel() {
        showStatus('📊 Generating Excel workbook...', 'info');
        try {
            const resp = await fetch('/api/export/excel', {method: 'POST'});
            const data = await resp.json();
            if (data.status === 'ok') {
                showStatus('📊 Excel exported: <strong>' + data.filename + '</strong>', 'success');
            } else {
                showStatus('❌ Export failed: ' + data.message, 'error');
            }
        } catch (e) { showStatus('❌ Export failed: ' + e.message, 'error'); }
    }

    async function exportCsv() {
        showStatus('📄 Exporting CSV...', 'info');
        try {
            const days = document.getElementById('daysSlider').value;
            const resp = await fetch('/api/export/csv?days=' + days, {method: 'POST'});
            const data = await resp.json();
            if (data.status === 'ok') {
                showStatus('📄 CSV exported: <strong>' + data.filename + '</strong>', 'success');
            } else {
                showStatus('❌ Export failed: ' + data.message, 'error');
            }
        } catch (e) { showStatus('❌ Export failed: ' + e.message, 'error'); }
    }

    loadKeywords();
</script>
</body>
</html>'''

with open(os.path.join(TEMPLATES_DIR, "index.html"), "w") as f:
    f.write(html_code)

print(f"✅ templates/index.html written ({len(html_code):,} characters)")

✅ templates/index.html written (16,234 characters)


In [8]:
# Cell 6: Write app.py — the main entry point

app_code = r'''"""
app.py — Grant Monitor Desktop Application
Launch: python app.py
"""

import threading
import json
import os
import sys
import webbrowser
from flask import Flask, render_template, request, jsonify

# Ensure we can find our modules
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
os.chdir(BASE_DIR)
sys.path.insert(0, BASE_DIR)

import grant_engine as engine

engine.init_db()

# ==============================================================
# FLASK APP
# ==============================================================

flask_app = Flask(
    __name__,
    template_folder=os.path.join(BASE_DIR, "templates"),
)
flask_app.config["SECRET_KEY"] = os.urandom(24)


@flask_app.route("/")
def index():
    return render_template("index.html")


# -- API: Keywords --

@flask_app.route("/api/keywords", methods=["GET"])
def api_get_keywords():
    return jsonify({"keywords": engine.get_active_keywords()})


@flask_app.route("/api/keywords", methods=["POST"])
def api_add_keyword():
    data = request.get_json()
    kw = data.get("keyword", "").strip()
    if not kw:
        return jsonify({"status": "error", "message": "Keyword cannot be empty."})
    engine.add_keyword(kw)
    return jsonify({"status": "ok", "message": f"Added: {kw}"})


@flask_app.route("/api/keywords", methods=["DELETE"])
def api_remove_keyword():
    data = request.get_json()
    kw = data.get("keyword", "").strip()
    if not kw:
        return jsonify({"status": "error", "message": "Keyword cannot be empty."})
    engine.remove_keyword(kw)
    return jsonify({"status": "ok", "message": f"Removed: {kw}"})


# -- API: Scan --

@flask_app.route("/api/scan", methods=["POST"])
def api_scan():
    try:
        summary = engine.run_full_scan()
        with engine.get_db() as conn:
            conn.execute("UPDATE grants SET notified = 1 WHERE notified = 0")
        return jsonify({"status": "ok", "summary": summary})
    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 500


# -- API: Get grants --

@flask_app.route("/api/grants", methods=["GET"])
def api_get_grants():
    days = int(request.args.get("days", 7))
    source = request.args.get("source", "all")
    df = engine.get_grants_df(since_days=days, source_filter=source)
    grants = []
    for _, row in df.iterrows():
        grants.append({
            "title": row["title"],
            "url": row["url"],
            "source": row["source"],
            "description": row.get("description", ""),
            "deadline": row.get("deadline", ""),
            "posted_date": row.get("posted_date", ""),
            "funding_amount": row.get("funding_amount", ""),
            "matched_keywords": row.get("matched_keywords", ""),
            "first_seen": row.get("first_seen", ""),
            "status": row.get("status", "NEW"),
        })
    return jsonify({"grants": grants, "count": len(grants)})


# -- API: Exports --

@flask_app.route("/api/export/excel", methods=["POST"])
def api_export_excel():
    try:
        filename = engine.generate_grant_excel()
        if filename:
            return jsonify({"status": "ok", "filename": filename})
        else:
            return jsonify({"status": "error", "message": "No grants in database."})
    except Exception as e:
        return jsonify({"status": "error", "message": str(e)})


@flask_app.route("/api/export/csv", methods=["POST"])
def api_export_csv():
    try:
        days = int(request.args.get("days", 7))
        filename = engine.export_csv(since_days=days)
        return jsonify({"status": "ok", "filename": filename})
    except Exception as e:
        return jsonify({"status": "error", "message": str(e)})


# ==============================================================
# LAUNCH
# ==============================================================

def main():
    USE_DESKTOP_WINDOW = True

    if USE_DESKTOP_WINDOW:
        try:
            import webview

            def start_flask():
                flask_app.run(host="127.0.0.1", port=5199,
                              debug=False, use_reloader=False)

            server = threading.Thread(target=start_flask, daemon=True)
            server.start()

            import time
            time.sleep(1.5)

            webview.create_window(
                title="Grant Monitor",
                url="http://127.0.0.1:5199",
                width=1200, height=850, min_size=(900, 600),
            )
            webview.start(debug=False)

        except ImportError:
            print("pywebview not found — falling back to browser mode")
            webbrowser.open("http://127.0.0.1:5199")
            flask_app.run(host="127.0.0.1", port=5199, debug=False)
    else:
        webbrowser.open("http://127.0.0.1:5199")
        flask_app.run(host="127.0.0.1", port=5199, debug=False)


if __name__ == "__main__":
    main()
'''

with open(os.path.join(PROJECT_DIR, "app.py"), "w") as f:
    f.write(app_code)

print("✅ app.py written")

✅ app.py written


In [9]:
# Cell 7: Verify the project structure

import os

print(f"📁 Project: {PROJECT_DIR}\n")
print("File structure:")
print("=" * 50)

for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, "").count(os.sep)
    indent = "   " * level
    folder_name = os.path.basename(root)
    print(f"{indent}📁 {folder_name}/")
    sub_indent = "   " * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        print(f"{sub_indent}📄 {file}  ({size:,} bytes)")

print("\n" + "=" * 50)
expected = ["app.py", "grant_engine.py", "requirements.txt"]
expected_sub = ["index.html"]

missing = []
for f in expected:
    if not os.path.exists(os.path.join(PROJECT_DIR, f)):
        missing.append(f)
for f in expected_sub:
    if not os.path.exists(os.path.join(TEMPLATES_DIR, f)):
        missing.append(f"templates/{f}")

if missing:
    print(f"❌ MISSING FILES: {missing}")
else:
    print("✅ All files present — ready to launch!")

📁 Project: /Users/kschro10/anaconda_projects/1b55a599-cf36-4af0-9a0e-86dc28da2d79

File structure:
📁 1b55a599-cf36-4af0-9a0e-86dc28da2d79/
   📄 project_folder_structure.ipynb  (137,598 bytes)
   📄 requirements.txt  (88 bytes)
   📄 ~$Grant_Tracker_20260428.xlsx  (165 bytes)
   📄 grant_monitor.log  (13,140 bytes)
   📄 .gitignore  (182 bytes)
   📄 app.py  (4,856 bytes)
   📄 Grant_Tracker_20260428.xlsx  (13,543 bytes)
   📄 grants.db  (65,536 bytes)
   📄 grant_engine.py  (23,049 bytes)
   📁 __pycache__/
      📄 grant_engine.cpython-313.pyc  (47,709 bytes)
   📁 templates/
      📄 index.html  (16,334 bytes)
   📁 .ipynb_checkpoints/
      📄 project_folder_structure-checkpoint.ipynb  (77,148 bytes)
   📁 .git/
      📄 config  (318 bytes)
      📄 HEAD  (21 bytes)
      📄 description  (73 bytes)
      📄 index  (766 bytes)
      📄 COMMIT_EDITMSG  (31 bytes)
      📁 objects/
         📁 61/
            📄 7d93cc5354d24dd08e09e4d306e0110d401137  (36,103 bytes)
         📁 57/
            📄 e3c8425c8d298

In [10]:
# Cell 8: Launch the Grant Monitor app
# 
# This opens the app in your web browser.
# The app runs until you stop this cell (click the ⏹ stop button).
#
# Your browser will open to: http://127.0.0.1:5199

import subprocess
import sys
import webbrowser
import time

print("🚀 Launching Grant Monitor...")
print(f"   Project: {PROJECT_DIR}")
print(f"   URL: http://127.0.0.1:5199")
print()
print("⚠️  To stop the app: click the ⏹ (stop/interrupt) button in the toolbar above")
print("=" * 60)

# Launch app.py as a subprocess from the project directory
process = subprocess.Popen(
    [sys.executable, "app.py"],
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for Flask to start, then open browser
time.sleep(3)
webbrowser.open("http://127.0.0.1:5199")

# Stream the output so you can see logs
try:
    for line in process.stdout:
        print(line, end="")
except KeyboardInterrupt:
    process.terminate()
    print("\n🛑 App stopped.")

🚀 Launching Grant Monitor...
   Project: /Users/kschro10/anaconda_projects/1b55a599-cf36-4af0-9a0e-86dc28da2d79
   URL: http://127.0.0.1:5199

⚠️  To stop the app: click the ⏹ (stop/interrupt) button in the toolbar above
2026-04-28 12:18:07,674 [INFO] Database initialized.
 * Serving Flask app 'app'
 * Debug mode: off
2026-04-28 12:18:07,699 [INFO] WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5199
2026-04-28 12:18:07,699 [INFO] Press CTRL+C to quit
2026-04-28 12:18:09,610 [INFO] 127.0.0.1 - - [28/Apr/2026 12:18:09] "GET / HTTP/1.1" 200 -
2026-04-28 12:18:09,617 [INFO] 127.0.0.1 - - [28/Apr/2026 12:18:09] "GET /api/keywords HTTP/1.1" 200 -
2026-04-28 12:18:09,944 [INFO] 127.0.0.1 - - [28/Apr/2026 12:18:09] "GET / HTTP/1.1" 200 -
2026-04-28 12:18:09,949 [INFO] 127.0.0.1 - - [28/Apr/2026 12:18:09] "GET /api/keywords HTTP/1.1" 200 -
2026-04-28 12:18:15,642 [INFO] 127.0.0.1 - - [28/Apr/2

In [11]:
# Verify all files are present

print(f"📁 Project: {PROJECT_DIR}\n")

expected_files = [
    "app.py",
    "grant_engine.py",
    "requirements.txt",
    os.path.join("templates", "index.html"),
]

all_good = True
for f in expected_files:
    full_path = os.path.join(PROJECT_DIR, f)
    exists = os.path.exists(full_path)
    size = os.path.getsize(full_path) if exists else 0
    icon = "✅" if exists else "❌"
    print(f"   {icon} {f:<30} {size:>8,} bytes")
    if not exists:
        all_good = False

print()
if all_good:
    print("✅ All files present — ready to launch!")
else:
    print("❌ Some files still missing — check above")

📁 Project: /Users/kschro10/anaconda_projects/1b55a599-cf36-4af0-9a0e-86dc28da2d79

   ✅ app.py                            4,856 bytes
   ✅ grant_engine.py                  23,049 bytes
   ✅ requirements.txt                     88 bytes
   ✅ templates/index.html             16,334 bytes

✅ All files present — ready to launch!


In [12]:
# Rewrite grant_engine.py with the FULL Excel export from your original notebook

grant_engine_code = r'''"""
grant_engine.py — All grant scanning, DB, keyword, and export logic.
Consolidated from the Jupyter notebook into a single importable module.
Includes full two-sheet Excel export with Calendar View.
"""

import requests
import feedparser
import sqlite3
import hashlib
import json
import re
import time
import logging
from bs4 import BeautifulSoup
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from contextlib import contextmanager
import pandas as pd
from datetime import datetime, timedelta, timezone

try:
    from dateutil.relativedelta import relativedelta
    HAS_DATEUTIL = True
except ImportError:
    HAS_DATEUTIL = False

try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter
    from openpyxl.worksheet.datavalidation import DataValidation
    HAS_OPENPYXL = True
except ImportError:
    HAS_OPENPYXL = False


# ==============================================================
# UTILITIES
# ==============================================================

def utc_now() -> datetime:
    return datetime.now(timezone.utc).replace(tzinfo=None)


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("grant_monitor.log"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("grant_monitor")

CONFIG = {
    "db_path": "grants.db",
    "request_delay_seconds": 2,
    "request_timeout_seconds": 30,
    "user_agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

DEFAULT_KEYWORDS = [
    "cybersecurity", "AI", "experiential learning", "medical technology",
    "data science", "machine learning", "artificial intelligence",
    "STEM education", "novel education", "workforce",
    "workforce development", "quantum",
]


# ==============================================================
# DATABASE
# ==============================================================

@contextmanager
def get_db(db_path: str = None):
    if db_path is None:
        db_path = CONFIG["db_path"]
    conn = sqlite3.connect(db_path, timeout=10)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA busy_timeout=5000")
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


@dataclass
class Grant:
    title: str
    url: str
    source: str
    description: str = ""
    deadline: Optional[str] = None
    posted_date: Optional[str] = None
    funding_amount: Optional[str] = None
    matched_keywords: list = field(default_factory=list)
    content_hash: str = ""
    first_seen: str = ""
    last_seen: str = ""
    is_new: bool = True

    def compute_hash(self):
        raw = f"{self.title}|{self.description}|{self.deadline}".lower().strip()
        self.content_hash = hashlib.sha256(raw.encode()).hexdigest()
        return self.content_hash


def init_db(db_path: str = None):
    if db_path is None:
        db_path = CONFIG["db_path"]
    with get_db(db_path) as conn:
        c = conn.cursor()
        c.execute("""
            CREATE TABLE IF NOT EXISTS grants (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL, url TEXT NOT NULL, source TEXT NOT NULL,
                description TEXT, deadline TEXT, posted_date TEXT,
                funding_amount TEXT, matched_keywords TEXT,
                content_hash TEXT UNIQUE, first_seen TEXT NOT NULL,
                last_seen TEXT NOT NULL, notified INTEGER DEFAULT 0
            )
        """)
        c.execute("""
            CREATE TABLE IF NOT EXISTS scan_log (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                source TEXT NOT NULL, scan_time TEXT NOT NULL,
                grants_found INTEGER DEFAULT 0, new_grants INTEGER DEFAULT 0,
                updated_grants INTEGER DEFAULT 0, status TEXT DEFAULT 'success',
                error_message TEXT
            )
        """)
        c.execute("""
            CREATE TABLE IF NOT EXISTS keywords (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                keyword TEXT UNIQUE NOT NULL,
                added_date TEXT NOT NULL, active INTEGER DEFAULT 1
            )
        """)
        now = utc_now().strftime("%Y-%m-%dT%H:%M:%S")
        for kw in DEFAULT_KEYWORDS:
            c.execute(
                "INSERT OR IGNORE INTO keywords (keyword, added_date) VALUES (?, ?)",
                (kw, now),
            )
    logger.info("Database initialized.")


def upsert_grant(grant: Grant, db_path: str = None) -> str:
    if db_path is None:
        db_path = CONFIG["db_path"]
    now = utc_now().strftime("%Y-%m-%dT%H:%M:%S")
    grant.compute_hash()
    grant.last_seen = now

    with get_db(db_path) as conn:
        c = conn.cursor()
        c.execute("SELECT id, content_hash FROM grants WHERE url = ?", (grant.url,))
        row = c.fetchone()

        if row is None:
            grant.first_seen = now
            c.execute("""
                INSERT INTO grants
                (title, url, source, description, deadline, posted_date,
                 funding_amount, matched_keywords, content_hash, first_seen, last_seen)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                grant.title, grant.url, grant.source, grant.description,
                grant.deadline, grant.posted_date, grant.funding_amount,
                json.dumps(grant.matched_keywords), grant.content_hash,
                grant.first_seen, grant.last_seen,
            ))
            return "new"
        elif row[1] != grant.content_hash:
            c.execute("""
                UPDATE grants SET title=?, description=?, deadline=?, posted_date=?,
                    funding_amount=?, matched_keywords=?, content_hash=?,
                    last_seen=?, notified=0
                WHERE url=?
            """, (
                grant.title, grant.description, grant.deadline,
                grant.posted_date, grant.funding_amount,
                json.dumps(grant.matched_keywords), grant.content_hash,
                grant.last_seen, grant.url,
            ))
            return "updated"
        else:
            c.execute("UPDATE grants SET last_seen=? WHERE url=?", (now, grant.url))
            return "unchanged"


def log_scan(source, grants_found, new_grants, updated_grants,
             status="success", error_message=None):
    with get_db() as conn:
        conn.execute("""
            INSERT INTO scan_log
            (source, scan_time, grants_found, new_grants, updated_grants, status, error_message)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (source, utc_now().strftime("%Y-%m-%dT%H:%M:%S"), grants_found,
              new_grants, updated_grants, status, error_message))


# ==============================================================
# KEYWORDS
# ==============================================================

def get_active_keywords(db_path: str = None) -> list:
    if db_path is None:
        db_path = CONFIG["db_path"]
    with get_db(db_path) as conn:
        c = conn.cursor()
        c.execute("SELECT keyword FROM keywords WHERE active = 1")
        return [row[0] for row in c.fetchall()]


def add_keyword(kw: str):
    now = utc_now().strftime("%Y-%m-%dT%H:%M:%S")
    with get_db() as conn:
        conn.execute(
            "INSERT OR IGNORE INTO keywords (keyword, added_date, active) VALUES (?, ?, 1)",
            (kw, now),
        )
        conn.execute(
            "UPDATE keywords SET active = 1 WHERE keyword = ? AND active = 0", (kw,)
        )


def remove_keyword(kw: str):
    with get_db() as conn:
        conn.execute("UPDATE keywords SET active = 0 WHERE keyword = ?", (kw,))


def match_keywords(text: str, keywords: list = None) -> list:
    if keywords is None:
        keywords = get_active_keywords()
    text_lower = text.lower()
    matched = []
    for kw in keywords:
        if kw == "AI":
            if re.search(r'\bAI\b', text):
                matched.append(kw)
        else:
            if re.search(r'\b' + re.escape(kw.lower()) + r'\b', text_lower):
                matched.append(kw)
    return matched


def passes_filter(title: str, description: str) -> list:
    return match_keywords(f"{title} {description}")


# ==============================================================
# HTTP SESSION
# ==============================================================

def get_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({
        "User-Agent": CONFIG["user_agent"],
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    })
    return s


SESSION = get_session()


def safe_request(url: str, **kwargs):
    try:
        time.sleep(CONFIG["request_delay_seconds"])
        resp = SESSION.get(url, timeout=CONFIG["request_timeout_seconds"], **kwargs)
        resp.raise_for_status()
        return resp
    except requests.RequestException as e:
        logger.error(f"Request failed for {url}: {e}")
        return None


# ==============================================================
# SCRAPERS
# ==============================================================

def scrape_grants_gov(keywords=None):
    if keywords is None:
        keywords = get_active_keywords()
    grants = []
    base_url = "https://apply07.grants.gov/grantsws/rest/opportunities/search/"
    for kw in keywords:
        payload = {
            "keyword": kw, "oppStatuses": "forecasted|posted",
            "sortBy": "openDate|desc", "rows": 50, "startRecordNum": 0,
        }
        try:
            time.sleep(CONFIG["request_delay_seconds"])
            resp = requests.post(base_url, json=payload,
                                 headers={"Content-Type": "application/json"},
                                 timeout=CONFIG["request_timeout_seconds"])
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            logger.error(f"Grants.gov API error for '{kw}': {e}")
            continue
        opportunities = data.get("oppHits", [])
        logger.info(f"Grants.gov: '{kw}' returned {len(opportunities)} results")
        for opp in opportunities:
            title = opp.get("title", "")
            description = opp.get("description", title)
            url = f"https://www.grants.gov/search-results-detail/{opp.get('id', '')}"
            matched = passes_filter(title, description)
            if matched:
                g = Grant(
                    title=title, url=url, source="grants.gov",
                    description=description[:2000],
                    deadline=opp.get("closeDate", ""),
                    posted_date=opp.get("openDate", ""),
                    funding_amount=opp.get("awardCeiling", ""),
                    matched_keywords=matched,
                )
                grants.append(g)
    seen = set()
    unique = []
    for g in grants:
        if g.url not in seen:
            seen.add(g.url)
            unique.append(g)
    logger.info(f"Grants.gov total unique matches: {len(unique)}")
    return unique


def scrape_nsf():
    feeds = [
        "https://www.nsf.gov/rss/rss_www_funding.xml",
        "https://www.nsf.gov/rss/rss_www_funding_pgm_annc_cise.xml",
        "https://www.nsf.gov/rss/rss_www_funding_pgm_annc_ehr.xml",
    ]
    grants = []
    for feed_url in feeds:
        try:
            time.sleep(CONFIG["request_delay_seconds"])
            feed = feedparser.parse(feed_url)
            logger.info(f"NSF feed: {len(feed.entries)} entries from {feed_url}")
        except Exception as e:
            logger.error(f"NSF feed error ({feed_url}): {e}")
            continue
        for entry in feed.entries:
            title = entry.get("title", "")
            summary = entry.get("summary", "")
            link = entry.get("link", "")
            published = entry.get("published", "")
            matched = passes_filter(title, summary)
            if matched:
                g = Grant(title=title, url=link, source="nsf.gov",
                          description=summary[:2000], posted_date=published,
                          matched_keywords=matched)
                grants.append(g)
    seen = set()
    unique = []
    for g in grants:
        if g.url not in seen:
            seen.add(g.url)
            unique.append(g)
    logger.info(f"NSF total unique matches: {len(unique)}")
    return unique


def scrape_nih_reporter():
    grants = []
    keywords = get_active_keywords()
    url = "https://api.reporter.nih.gov/v2/projects/search"
    search_text = " OR ".join(
        [kw for kw in keywords if kw.lower() in
         ("medical technology", "ai", "artificial intelligence",
          "data science", "machine learning", "cybersecurity")]
    )
    payload = {
        "criteria": {
            "advanced_text_search": {
                "operator": "or", "search_field": "projecttitle,terms",
                "search_text": search_text,
            },
            "fiscal_years": [2025, 2026],
            "newly_added_projects_only": True,
        },
        "offset": 0, "limit": 50,
        "sort_field": "project_start_date", "sort_order": "desc",
    }
    try:
        time.sleep(CONFIG["request_delay_seconds"])
        resp = requests.post(url, json=payload, timeout=CONFIG["request_timeout_seconds"])
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        logger.error(f"NIH Reporter API error: {e}")
        return grants
    results = data.get("results", [])
    logger.info(f"NIH Reporter: {len(results)} results")
    for proj in results:
        title = proj.get("project_title", "")
        abstract = proj.get("abstract_text", "") or ""
        link = f"https://reporter.nih.gov/project-details/{proj.get('application_id', '')}"
        matched = passes_filter(title, abstract)
        if matched:
            award = proj.get("award_amount")
            g = Grant(
                title=title.strip(), url=link, source="nih_reporter",
                description=abstract[:2000],
                posted_date=proj.get("project_start_date", ""),
                funding_amount=str(award) if award else "",
                matched_keywords=matched,
            )
            grants.append(g)
    logger.info(f"NIH Reporter matches: {len(grants)}")
    return grants


def scrape_rss_source(feed_url, source_name):
    grants = []
    try:
        time.sleep(CONFIG["request_delay_seconds"])
        feed = feedparser.parse(feed_url)
        logger.info(f"{source_name}: {len(feed.entries)} entries")
    except Exception as e:
        logger.error(f"{source_name} feed error: {e}")
        return grants
    for entry in feed.entries:
        title = entry.get("title", "")
        summary = entry.get("summary", entry.get("description", ""))
        link = entry.get("link", "")
        published = entry.get("published", entry.get("updated", ""))
        matched = passes_filter(title, summary)
        if matched:
            g = Grant(title=title, url=link, source=source_name,
                      description=summary[:2000], posted_date=published,
                      matched_keywords=matched)
            grants.append(g)
    logger.info(f"{source_name} matches: {len(grants)}")
    return grants


ADDITIONAL_RSS_SOURCES = [
    {"url": "https://www.ed.gov/feed", "name": "ed.gov"},
    {"url": "https://www.sbir.gov/rss-feed", "name": "sbir.gov"},
    {"url": "https://www.darpa.mil/rss", "name": "darpa.mil"},
]


# ==============================================================
# FULL SCAN
# ==============================================================

def run_full_scan() -> dict:
    logger.info("=" * 60)
    logger.info("STARTING FULL GRANT SCAN")
    logger.info(f"Active keywords: {get_active_keywords()}")
    logger.info("=" * 60)

    summary = {
        "scan_time": utc_now().isoformat(),
        "sources_scanned": 0, "total_found": 0,
        "new": 0, "updated": 0, "unchanged": 0, "errors": [],
    }
    all_grants = []

    try:
        results = scrape_grants_gov()
        all_grants.extend(results)
        log_scan("grants.gov", len(results), 0, 0)
        summary["sources_scanned"] += 1
    except Exception as e:
        logger.error(f"Grants.gov scraper failed: {e}")
        log_scan("grants.gov", 0, 0, 0, "error", str(e))
        summary["errors"].append(f"grants.gov: {e}")

    try:
        results = scrape_nsf()
        all_grants.extend(results)
        log_scan("nsf.gov", len(results), 0, 0)
        summary["sources_scanned"] += 1
    except Exception as e:
        logger.error(f"NSF scraper failed: {e}")
        log_scan("nsf.gov", 0, 0, 0, "error", str(e))
        summary["errors"].append(f"nsf.gov: {e}")

    try:
        results = scrape_nih_reporter()
        all_grants.extend(results)
        log_scan("nih_reporter", len(results), 0, 0)
        summary["sources_scanned"] += 1
    except Exception as e:
        logger.error(f"NIH Reporter scraper failed: {e}")
        log_scan("nih_reporter", 0, 0, 0, "error", str(e))
        summary["errors"].append(f"nih_reporter: {e}")

    for src in ADDITIONAL_RSS_SOURCES:
        try:
            results = scrape_rss_source(src["url"], src["name"])
            all_grants.extend(results)
            log_scan(src["name"], len(results), 0, 0)
            summary["sources_scanned"] += 1
        except Exception as e:
            logger.error(f"{src['name']} scraper failed: {e}")
            log_scan(src["name"], 0, 0, 0, "error", str(e))
            summary["errors"].append(f"{src['name']}: {e}")

    summary["total_found"] = len(all_grants)
    for grant in all_grants:
        result = upsert_grant(grant)
        summary[result] = summary.get(result, 0) + 1

    logger.info("=" * 60)
    logger.info(f"SCAN COMPLETE: {summary}")
    logger.info("=" * 60)
    return summary


# ==============================================================
# REPORTS
# ==============================================================

def get_grants_df(since_days: int = 7, source_filter: str = None) -> pd.DataFrame:
    cutoff = (utc_now() - timedelta(days=since_days)).strftime("%Y-%m-%dT%H:%M:%S")
    with get_db() as conn:
        df = pd.read_sql_query("""
            SELECT title, url, source, description, deadline,
                   posted_date, funding_amount, matched_keywords,
                   first_seen, last_seen,
                   CASE WHEN first_seen >= ? THEN 'NEW' ELSE 'UPDATED' END as status
            FROM grants
            WHERE first_seen >= ? OR (last_seen >= ? AND notified = 0)
            ORDER BY first_seen DESC
        """, conn, params=(cutoff, cutoff, cutoff))
    if not df.empty:
        df["matched_keywords"] = df["matched_keywords"].apply(
            lambda x: ", ".join(json.loads(x)) if x else ""
        )
    if source_filter and source_filter != "all":
        df = df[df["source"] == source_filter]
    return df


def get_all_grants_df() -> pd.DataFrame:
    with get_db() as conn:
        df = pd.read_sql_query("""
            SELECT title, url, source, description, deadline,
                   posted_date, funding_amount, matched_keywords,
                   first_seen, last_seen
            FROM grants
            ORDER BY
                CASE WHEN deadline IS NOT NULL AND deadline != '' THEN 0 ELSE 1 END,
                deadline ASC
        """, conn)
    if not df.empty:
        df["matched_keywords"] = df["matched_keywords"].apply(
            lambda x: json.loads(x) if x else []
        )
    return df


def export_csv(since_days: int = 7) -> str:
    df = get_grants_df(since_days)
    filename = f"grant_report_{utc_now().strftime('%Y%m%d')}.csv"
    df.to_csv(filename, index=False)
    logger.info(f"CSV exported: {filename}")
    return filename


# ==============================================================
# EXCEL EXPORT — FULL TWO-SHEET VERSION
# ==============================================================

def parse_deadline(raw: str) -> Optional[datetime]:
    if not raw or raw.strip() == "":
        return None
    formats = [
        "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d", "%m/%d/%Y",
        "%B %d, %Y", "%b %d, %Y", "%m-%d-%Y", "%Y%m%d",
    ]
    cleaned = raw.strip().split("+")[0].split("Z")[0]
    for fmt in formats:
        try:
            return datetime.strptime(cleaned, fmt)
        except ValueError:
            continue
    return None


def infer_grant_type(grant_row: dict) -> str:
    deadline = str(grant_row.get("deadline", "") or "")
    posted = str(grant_row.get("posted_date", "") or "")
    title = (str(grant_row.get("title", "")) + " " + str(grant_row.get("description", ""))).lower()
    now = utc_now()

    if any(term in title for term in ["letter of intent", "loi", "pre-application"]):
        return "LOI"
    if any(term in title for term in ["rolling", "continuous", "ongoing", "no deadline"]):
        return "Ongoing"

    dl = parse_deadline(deadline)
    if dl:
        days_until = (dl - now).days
        if days_until < 0:
            return "Monitoring"
        elif days_until > 180:
            return "Future"
        else:
            return "Deadline"

    if any(term in title for term in ["forecast", "anticipated", "expected", "upcoming"]):
        return "Future"
    if posted and not deadline:
        return "Monitoring"
    return "Monitoring"


def infer_confidence(grant_row: dict) -> str:
    deadline = str(grant_row.get("deadline", "") or "")
    title = (str(grant_row.get("title", "")) + " " + str(grant_row.get("description", ""))).lower()

    if any(term in title for term in ["rolling", "continuous"]):
        return "Rolling"
    if any(term in title for term in ["annual", "yearly", "recurring", "periodic"]):
        return "Periodic"
    if any(term in title for term in ["forecast", "anticipated", "expected"]):
        return "Future"

    dl = parse_deadline(deadline)
    if dl:
        return "Confirmed"

    if any(term in title for term in ["estimated", "approximate", "tentative"]):
        return "EST"
    return "Likely"


def extract_funder(source: str, title: str, description: str) -> str:
    source_map = {
        "grants.gov": "Federal (see listing)",
        "nsf.gov": "National Science Foundation",
        "nih_reporter": "National Institutes of Health",
        "ed.gov": "U.S. Department of Education",
        "sbir.gov": "SBIR/STTR Program",
        "darpa.mil": "DARPA",
    }
    funder = source_map.get(source, "")
    combined = f"{title} {description}".lower()
    agency_patterns = {
        "national science foundation": "National Science Foundation (NSF)",
        "nsf": "National Science Foundation (NSF)",
        "department of defense": "Department of Defense (DoD)",
        "dod": "Department of Defense (DoD)",
        "department of energy": "Department of Energy (DOE)",
        "doe": "Department of Energy (DOE)",
        "department of education": "U.S. Department of Education",
        "national institutes of health": "National Institutes of Health (NIH)",
        "nih": "National Institutes of Health (NIH)",
        "darpa": "DARPA",
        "nasa": "NASA",
        "nist": "National Institute of Standards and Technology (NIST)",
        "department of transportation": "Department of Transportation (DOT)",
    }
    for pattern, agency in agency_patterns.items():
        if pattern in combined:
            return agency
    return funder if funder else "See listing"


def generate_grant_excel(output_filename: str = None) -> str:
    if not HAS_OPENPYXL:
        raise ImportError("openpyxl is required. pip install openpyxl")
    if not HAS_DATEUTIL:
        raise ImportError("python-dateutil is required. pip install python-dateutil")
    if output_filename is None:
        output_filename = f"Grant_Tracker_{utc_now().strftime('%Y%m%d')}.xlsx"

    df = get_all_grants_df()
    if df.empty:
        return None

    wb = openpyxl.Workbook()

    # ── Styling constants ──
    HEADER_FILL = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
    HEADER_FONT = Font(name="Calibri", bold=True, color="FFFFFF", size=11)
    THIN_BORDER = Border(
        left=Side(style="thin", color="B4C6E7"),
        right=Side(style="thin", color="B4C6E7"),
        top=Side(style="thin", color="B4C6E7"),
        bottom=Side(style="thin", color="B4C6E7"),
    )
    TYPE_COLORS = {
        "LOI": PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid"),
        "Deadline": PatternFill(start_color="FCE4EC", end_color="FCE4EC", fill_type="solid"),
        "Opens": PatternFill(start_color="E8F5E9", end_color="E8F5E9", fill_type="solid"),
        "Future": PatternFill(start_color="E3F2FD", end_color="E3F2FD", fill_type="solid"),
        "Monitoring": PatternFill(start_color="F3E5F5", end_color="F3E5F5", fill_type="solid"),
        "Ongoing": PatternFill(start_color="E0F2F1", end_color="E0F2F1", fill_type="solid"),
    }
    CONFIDENCE_COLORS = {
        "Confirmed": Font(color="2E7D32", bold=True),
        "EST": Font(color="F57F17"),
        "Future": Font(color="1565C0"),
        "Likely": Font(color="6A1B9A"),
        "Periodic": Font(color="00695C"),
        "Rolling": Font(color="BF360C"),
    }
    CAL_DEADLINE = PatternFill(start_color="E53935", end_color="E53935", fill_type="solid")
    CAL_LOI = PatternFill(start_color="FFB300", end_color="FFB300", fill_type="solid")

    # ==========================================================
    # SHEET 1: GRANT TRACKER
    # ==========================================================

    ws1 = wb.active
    ws1.title = "Grant Tracker"
    ws1.sheet_properties.tabColor = "1F4E79"
    ws1.freeze_panes = "A2"

    tracker_columns = [
        ("Deadline Date", 16), ("Type", 14), ("Focus Area", 28),
        ("Grant / Program", 50), ("Funder", 32), ("Estimated Amount", 20),
        ("Confidence", 14), ("Action Required", 35), ("URL", 45),
    ]

    for col_idx, (header, width) in enumerate(tracker_columns, 1):
        cell = ws1.cell(row=1, column=col_idx, value=header)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = THIN_BORDER
        ws1.column_dimensions[get_column_letter(col_idx)].width = width

    ws1.row_dimensions[1].height = 30

    for row_idx, (_, grant) in enumerate(df.iterrows(), 2):
        grant_dict = grant.to_dict()
        kw_list = grant["matched_keywords"] if isinstance(grant["matched_keywords"], list) else []

        # Deadline Date
        dl = parse_deadline(str(grant.get("deadline", "") or ""))
        deadline_cell = ws1.cell(row=row_idx, column=1)
        if dl:
            deadline_cell.value = dl
            deadline_cell.number_format = "MMM DD, YYYY"
            days_left = (dl - utc_now()).days
            if 0 < days_left <= 30:
                deadline_cell.font = Font(color="C62828", bold=True)
            elif 0 < days_left <= 60:
                deadline_cell.font = Font(color="E65100")
        else:
            deadline_cell.value = "TBD"
            deadline_cell.font = Font(color="9E9E9E", italic=True)
        deadline_cell.alignment = Alignment(horizontal="center")
        deadline_cell.border = THIN_BORDER

        # Type
        grant_type = infer_grant_type(grant_dict)
        type_cell = ws1.cell(row=row_idx, column=2, value=grant_type)
        type_cell.alignment = Alignment(horizontal="center")
        type_cell.fill = TYPE_COLORS.get(grant_type, PatternFill())
        type_cell.border = THIN_BORDER

        # Focus Area
        focus = ", ".join(kw_list)
        focus_cell = ws1.cell(row=row_idx, column=3, value=focus)
        focus_cell.alignment = Alignment(wrap_text=True, vertical="top")
        focus_cell.border = THIN_BORDER

        # Grant / Program
        title_cell = ws1.cell(row=row_idx, column=4, value=grant["title"])
        title_cell.alignment = Alignment(wrap_text=True, vertical="top")
        title_cell.font = Font(name="Calibri", size=10)
        title_cell.border = THIN_BORDER

        # Funder
        funder = extract_funder(grant["source"], grant["title"],
                                str(grant.get("description", "") or ""))
        funder_cell = ws1.cell(row=row_idx, column=5, value=funder)
        funder_cell.alignment = Alignment(wrap_text=True, vertical="top")
        funder_cell.border = THIN_BORDER

        # Estimated Amount
        amount_cell = ws1.cell(row=row_idx, column=6)
        raw_amount = str(grant.get("funding_amount", "") or "").strip()
        try:
            numeric = float(re.sub(r'[^\d.]', '', raw_amount))
            if numeric > 0:
                amount_cell.value = numeric
                amount_cell.number_format = '$#,##0'
            else:
                amount_cell.value = "See listing"
                amount_cell.font = Font(color="9E9E9E", italic=True)
        except (ValueError, TypeError):
            amount_cell.value = raw_amount if raw_amount else "See listing"
            if not raw_amount:
                amount_cell.font = Font(color="9E9E9E", italic=True)
        amount_cell.alignment = Alignment(horizontal="center")
        amount_cell.border = THIN_BORDER

        # Confidence
        confidence = infer_confidence(grant_dict)
        conf_cell = ws1.cell(row=row_idx, column=7, value=confidence)
        conf_cell.font = CONFIDENCE_COLORS.get(confidence, Font())
        conf_cell.alignment = Alignment(horizontal="center")
        conf_cell.border = THIN_BORDER

        # Action Required
        action = ""
        if grant_type == "Deadline":
            if dl:
                days_left = (dl - utc_now()).days
                if days_left <= 14:
                    action = "URGENT: Submit application"
                elif days_left <= 30:
                    action = "Finalize application materials"
                elif days_left <= 60:
                    action = "Begin application drafting"
                else:
                    action = "Review requirements; plan timeline"
            else:
                action = "Review requirements"
        elif grant_type == "LOI":
            action = "Prepare Letter of Intent"
        elif grant_type == "Future":
            action = "Monitor for announcement"
        elif grant_type == "Monitoring":
            action = "Watch for next cycle"
        elif grant_type == "Ongoing":
            action = "Review eligibility; apply when ready"
        else:
            action = "Review opportunity"
        action_cell = ws1.cell(row=row_idx, column=8, value=action)
        action_cell.alignment = Alignment(wrap_text=True, vertical="top")
        action_cell.border = THIN_BORDER

        # URL
        url_cell = ws1.cell(row=row_idx, column=9, value=grant["url"])
        url_cell.font = Font(color="0563C1", underline="single", size=9)
        url_cell.hyperlink = grant["url"]
        url_cell.alignment = Alignment(wrap_text=True, vertical="top")
        url_cell.border = THIN_BORDER

        ws1.row_dimensions[row_idx].height = 35

    # Autofilter
    ws1.auto_filter.ref = f"A1:{get_column_letter(len(tracker_columns))}{len(df) + 1}"

    # Data validation dropdowns
    type_validation = DataValidation(
        type="list",
        formula1='"LOI,Deadline,Opens,Future,Monitoring,Ongoing"',
        allow_blank=True,
    )
    type_validation.error = "Please select a valid type"
    ws1.add_data_validation(type_validation)
    type_validation.add(f"B2:B{len(df) + 1}")

    conf_validation = DataValidation(
        type="list",
        formula1='"Confirmed,EST,Future,Likely,Periodic,Rolling"',
        allow_blank=True,
    )
    ws1.add_data_validation(conf_validation)
    conf_validation.add(f"G2:G{len(df) + 1}")

    # ==========================================================
    # SHEET 2: CALENDAR VIEW
    # ==========================================================

    ws2 = wb.create_sheet("Calendar View")
    ws2.sheet_properties.tabColor = "2E7D32"
    ws2.freeze_panes = "C2"

    now = utc_now()
    months = []
    for i in range(24):
        months.append(now + relativedelta(months=i))

    cal_columns = ["Focus Area", "Grant / Program"]
    for m in months:
        cal_columns.append(m.strftime("%b %Y"))
    cal_columns.append("Notes / Key Actions")

    ws2.column_dimensions["A"].width = 24
    ws2.column_dimensions["B"].width = 42
    for col_idx in range(3, 3 + 24):
        ws2.column_dimensions[get_column_letter(col_idx)].width = 12
    ws2.column_dimensions[get_column_letter(27)].width = 35

    for col_idx, header in enumerate(cal_columns, 1):
        cell = ws2.cell(row=1, column=col_idx, value=header)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = THIN_BORDER

    ws2.row_dimensions[1].height = 30

    for row_idx, (_, grant) in enumerate(df.iterrows(), 2):
        grant_dict = grant.to_dict()
        kw_list = grant["matched_keywords"] if isinstance(grant["matched_keywords"], list) else []

        # Focus Area
        focus = ", ".join(kw_list)
        focus_cell = ws2.cell(row=row_idx, column=1, value=focus)
        focus_cell.alignment = Alignment(wrap_text=True, vertical="center")
        focus_cell.border = THIN_BORDER

        # Grant / Program
        title_cell = ws2.cell(row=row_idx, column=2, value=grant["title"])
        title_cell.alignment = Alignment(wrap_text=True, vertical="center")
        title_cell.font = Font(name="Calibri", size=9)
        title_cell.border = THIN_BORDER

        # Month columns
        dl = parse_deadline(str(grant.get("deadline", "") or ""))
        grant_type = infer_grant_type(grant_dict)

        for month_idx, month_date in enumerate(months):
            col = month_idx + 3
            cell = ws2.cell(row=row_idx, column=col)
            cell.border = THIN_BORDER
            cell.alignment = Alignment(horizontal="center", vertical="center")

            if dl:
                if dl.year == month_date.year and dl.month == month_date.month:
                    if grant_type == "LOI":
                        cell.value = "LOI Due"
                        cell.fill = CAL_LOI
                        cell.font = Font(bold=True, size=9, color="FFFFFF")
                    else:
                        cell.value = "DEADLINE"
                        cell.fill = CAL_DEADLINE
                        cell.font = Font(bold=True, size=9, color="FFFFFF")

                prep_date = dl - relativedelta(months=1)
                if prep_date.year == month_date.year and prep_date.month == month_date.month:
                    cell.value = "Prepare"
                    cell.fill = PatternFill(start_color="FFF9C4", end_color="FFF9C4", fill_type="solid")
                    cell.font = Font(size=9, color="F57F17")

                plan_date = dl - relativedelta(months=2)
                if plan_date.year == month_date.year and plan_date.month == month_date.month:
                    cell.value = "Plan"
                    cell.fill = PatternFill(start_color="E3F2FD", end_color="E3F2FD", fill_type="solid")
                    cell.font = Font(size=9, color="1565C0")

            elif grant_type == "Ongoing":
                cell.value = "Open"
                cell.fill = PatternFill(start_color="E8F5E9", end_color="E8F5E9", fill_type="solid")
                cell.font = Font(size=8, color="2E7D32")

            elif grant_type == "Monitoring":
                if month_idx < 6:
                    cell.value = "Watch"
                    cell.fill = PatternFill(start_color="F5F5F5", end_color="F5F5F5", fill_type="solid")
                    cell.font = Font(size=8, color="9E9E9E")

        # Notes / Key Actions
        notes = ""
        if dl:
            days_left = (dl - utc_now()).days
            if days_left > 0:
                notes = f"{days_left} days until deadline"
            elif days_left == 0:
                notes = "DEADLINE TODAY"
            else:
                notes = f"Closed {abs(days_left)} days ago - watch for next cycle"
        elif grant_type == "Ongoing":
            notes = "Rolling - apply when ready"
        else:
            notes = "Monitor for updates"

        notes_cell = ws2.cell(row=row_idx, column=27, value=notes)
        notes_cell.alignment = Alignment(wrap_text=True, vertical="center")
        notes_cell.border = THIN_BORDER

        ws2.row_dimensions[row_idx].height = 30

    # Legend
    legend_start = len(df) + 3
    ws2.cell(row=legend_start, column=1, value="LEGEND").font = Font(bold=True, size=11)

    legend_items = [
        ("DEADLINE", CAL_DEADLINE, Font(bold=True, color="FFFFFF", size=9)),
        ("LOI Due", CAL_LOI, Font(bold=True, color="FFFFFF", size=9)),
        ("Prepare", PatternFill(start_color="FFF9C4", end_color="FFF9C4", fill_type="solid"),
         Font(size=9, color="F57F17")),
        ("Plan", PatternFill(start_color="E3F2FD", end_color="E3F2FD", fill_type="solid"),
         Font(size=9, color="1565C0")),
        ("Open (Rolling)", PatternFill(start_color="E8F5E9", end_color="E8F5E9", fill_type="solid"),
         Font(size=8, color="2E7D32")),
        ("Watch", PatternFill(start_color="F5F5F5", end_color="F5F5F5", fill_type="solid"),
         Font(size=8, color="9E9E9E")),
    ]

    for i, (label, fill, font) in enumerate(legend_items):
        row = legend_start + 1 + i
        sample = ws2.cell(row=row, column=1, value=label)
        sample.fill = fill
        sample.font = font
        sample.alignment = Alignment(horizontal="center")
        sample.border = THIN_BORDER
        ws2.cell(row=row, column=2, value=f"= {label}").font = Font(size=9, color="666666")

    # Save
    wb.save(output_filename)
    logger.info(f"Excel workbook saved: {output_filename}")

    file_size = Path(output_filename).stat().st_size / 1024
    logger.info(f"  Sheet 1 - Grant Tracker: {len(df)} grants")
    logger.info(f"  Sheet 2 - Calendar View: {len(df)} grants x 24 months")
    logger.info(f"  File size: {file_size:.1f} KB")

    return output_filename
'''

with open(os.path.join(PROJECT_DIR, "grant_engine.py"), "w") as f:
    f.write(grant_engine_code)

print(f"✅ grant_engine.py rewritten with full Excel export ({len(grant_engine_code):,} characters)")
print()
print("   Sheet 1: Grant Tracker")
print("     - Deadline Date (color-coded: red <30 days, orange <60 days)")
print("     - Type (LOI / Deadline / Future / Monitoring / Ongoing)")
print("     - Focus Area, Grant/Program, Funder, Estimated Amount")
print("     - Confidence (Confirmed / EST / Future / Likely / Periodic / Rolling)")
print("     - Action Required (auto-generated based on deadline proximity)")
print("     - URL (clickable hyperlink)")
print("     - Dropdown validation on Type and Confidence columns")
print()
print("   Sheet 2: Calendar View")
print("     - 24-month timeline with color-coded cells")
print("     - DEADLINE (red), LOI Due (amber), Prepare (yellow), Plan (blue)")
print("     - Open (green) for rolling, Watch (gray) for monitoring")
print("     - Notes column with days-until-deadline")
print("     - Legend at bottom")

✅ grant_engine.py rewritten with full Excel export (38,476 characters)

   Sheet 1: Grant Tracker
     - Deadline Date (color-coded: red <30 days, orange <60 days)
     - Type (LOI / Deadline / Future / Monitoring / Ongoing)
     - Focus Area, Grant/Program, Funder, Estimated Amount
     - Confidence (Confirmed / EST / Future / Likely / Periodic / Rolling)
     - Action Required (auto-generated based on deadline proximity)
     - URL (clickable hyperlink)
     - Dropdown validation on Type and Confidence columns

   Sheet 2: Calendar View
     - 24-month timeline with color-coded cells
     - DEADLINE (red), LOI Due (amber), Prepare (yellow), Plan (blue)
     - Open (green) for rolling, Watch (gray) for monitoring
     - Notes column with days-until-deadline
     - Legend at bottom


In [ ]:
# Cell 8: Launch the Grant Monitor app
# 
# This opens the app in your web browser.
# The app runs until you stop this cell (click the ⏹ stop button).
#
# Your browser will open to: http://127.0.0.1:5199

import subprocess
import sys
import webbrowser
import time

print("🚀 Launching Grant Monitor...")
print(f"   Project: {PROJECT_DIR}")
print(f"   URL: http://127.0.0.1:5199")
print()
print("⚠️  To stop the app: click the ⏹ (stop/interrupt) button in the toolbar above")
print("=" * 60)

# Launch app.py as a subprocess from the project directory
process = subprocess.Popen(
    [sys.executable, "app.py"],
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Wait for Flask to start, then open browser
time.sleep(3)
webbrowser.open("http://127.0.0.1:5199")

# Stream the output so you can see logs
try:
    for line in process.stdout:
        print(line, end="")
except KeyboardInterrupt:
    process.terminate()
    print("\n🛑 App stopped.")

🚀 Launching Grant Monitor...
   Project: /Users/kschro10/anaconda_projects/1b55a599-cf36-4af0-9a0e-86dc28da2d79
   URL: http://127.0.0.1:5199

⚠️  To stop the app: click the ⏹ (stop/interrupt) button in the toolbar above
2026-04-28 12:26:32,715 [INFO] Database initialized.
 * Serving Flask app 'app'
 * Debug mode: off
2026-04-28 12:26:32,743 [INFO] WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5199
2026-04-28 12:26:32,743 [INFO] Press CTRL+C to quit
2026-04-28 12:26:34,746 [INFO] 127.0.0.1 - - [28/Apr/2026 12:26:34] "GET / HTTP/1.1" 200 -
2026-04-28 12:26:34,753 [INFO] 127.0.0.1 - - [28/Apr/2026 12:26:34] "GET /api/keywords HTTP/1.1" 200 -
2026-04-28 12:26:34,909 [INFO] 127.0.0.1 - - [28/Apr/2026 12:26:34] "GET / HTTP/1.1" 200 -
2026-04-28 12:26:34,916 [INFO] 127.0.0.1 - - [28/Apr/2026 12:26:34] "GET /api/keywords HTTP/1.1" 200 -
2026-04-28 12:26:39,683 [INFO] 127.0.0.1 - - [28/Apr/2